# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krishamerja1903/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)
print("Connected as:", api.whoami()["name"])

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march = pd.read_parquet(file_path)

available = march[
    march["gsc_data_available"].eq(True) &
    march["ga4_data_available"].eq(True)
].copy()

print("March rows:", len(march))
print("Available rows:", len(available))
print("Unique pages:", available["content_hash_id"].nunique())

print("\nColumns:")
print(available.columns.tolist())

Connected as: Krisha1903
March rows: 9841378
Available rows: 364347
Unique pages: 63856

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


### Method choice

I use Logistic Regression as the first capstone model. The goal is to identify pages that are likely to show declining search performance using simple observed search and engagement features.

Logistic Regression is appropriate because the target is binary (declining vs not declining), the model is interpretable, and it provides predicted probabilities that can be used to rank pages. This also gives a transparent comparison with the simple Week-4 rule-based baseline without adding unnecessary model complexity.

The model is used for directional decision support rather than as evidence of causal search-ranking effects.

In [6]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

print("Method: Logistic Regression")
print("Purpose: binary decline prediction + ranked decision-support score")
print("Model configured successfully.")

Method: Logistic Regression
Purpose: binary decline prediction + ranked decision-support score
Model configured successfully.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Split design

I use a time-aware design within March 2026. Features are calculated from the early observation window (March 1–15), while the target is based on whether average daily clicks are lower in the later window (March 16–31).

This keeps future-period performance out of the model inputs. Pages must have observations in both periods so that the decline label can be calculated consistently.

The train/test split is grouped by client so pages from the same client do not appear in both sets. This reduces leakage from client-specific patterns and provides a more honest estimate of generalisation to unseen clients.

In [7]:
from sklearn.model_selection import GroupShuffleSplit

# Ensure correct date type
available["report_date"] = pd.to_datetime(available["report_date"])

# Early and late March windows
early = available[
    available["report_date"].between("2026-03-01", "2026-03-15")
].copy()

late = available[
    available["report_date"].between("2026-03-16", "2026-03-31")
].copy()

# Aggregate early-window features per page
early_page = (
    early.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        pageviews=("ga4_pageviews", "sum"),
        engaged_sessions=("ga4_engaged_sessions", "sum"),
        early_days=("report_date", "nunique")
    )
)

# Aggregate late-window clicks for the target
late_page = (
    late.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        late_clicks=("gsc_clicks", "sum"),
        late_days=("report_date", "nunique")
    )
)

# Keep pages observed in both windows
model_df = early_page.merge(
    late_page,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Compare average DAILY clicks so different day counts are handled honestly
model_df["early_avg_clicks"] = (
    model_df["clicks"] / model_df["early_days"]
)

model_df["late_avg_clicks"] = (
    model_df["late_clicks"] / model_df["late_days"]
)

# Target: 1 = declining
model_df["declining"] = (
    model_df["late_avg_clicks"] < model_df["early_avg_clicks"]
).astype(int)

# CTR from EARLY window only
model_df["ctr"] = np.where(
    model_df["impressions"] > 0,
    model_df["clicks"] / model_df["impressions"],
    0
)

features = [
    "impressions",
    "clicks",
    "avg_position",
    "pageviews",
    "engaged_sessions"
]

# Remove missing/infinite feature rows
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=features)

# Grouped train/test split by client
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        model_df["declining"],
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Model rows:", len(model_df))
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())
print(
    "Client overlap:",
    len(
        set(train_df["client_hash_id"])
        & set(test_df["client_hash_id"])
    )
)
print("Decline rate:", round(model_df["declining"].mean(), 4))

Model rows: 29394
Train rows: 28227
Test rows: 1167
Train clients: 22
Test clients: 6
Client overlap: 0
Decline rate: 0.3643


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Model vs baseline

I compare Logistic Regression with the Week-4 rule-based baseline on the same held-out test set.

Both methods rank the same test pages and are evaluated using Precision@50. This keeps the comparison consistent. The model uses only early-window features, while the decline outcome is measured from the later window.

The Week-4 baseline prioritises pages with relatively high visibility and low CTR. The model is considered useful only if it provides evidence of improvement over this simple benchmark on the same evaluation set.

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Features and target
X_train = train_df[features]
y_train = train_df["declining"]

X_test = test_df[features]
y_test = test_df["declining"]

# Logistic Regression pipeline
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

# Predicted probability of decline
test_df = test_df.copy()
test_df["model_score"] = model.predict_proba(X_test)[:, 1]

# -------------------------
# Week-4 baseline
# -------------------------

# Thresholds learned from TRAINING data only
imp_threshold = train_df["impressions"].median()
ctr_threshold = train_df["ctr"].median()

test_df["high_visibility"] = (
    test_df["impressions"] >= imp_threshold
).astype(int)

test_df["low_ctr"] = (
    test_df["ctr"] <= ctr_threshold
).astype(int)

test_df["baseline_score"] = (
    test_df["high_visibility"] * 50 +
    test_df["low_ctr"] * 50
)

# Use impressions only as the Week-4 tie-break
test_df["baseline_rank_score"] = (
    test_df["baseline_score"] * 1_000_000_000 +
    test_df["impressions"]
)

# -------------------------
# Precision@K
# -------------------------

def precision_at_k(df, score_col, k=50):
    top_k = df.sort_values(
        score_col,
        ascending=False
    ).head(k)

    return top_k["declining"].mean()

model_p50 = precision_at_k(
    test_df,
    "model_score",
    50
)

baseline_p50 = precision_at_k(
    test_df,
    "baseline_rank_score",
    50
)

comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "Precision@50": [
        baseline_p50,
        model_p50
    ]
})

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Positive test cases:", int(test_df["declining"].sum()))
print("\nMODEL VS BASELINE")
display(comparison)

print(
    "\nDifference:",
    round(model_p50 - baseline_p50, 4)
)

Training rows: 28227
Test rows: 1167
Positive test cases: 439

MODEL VS BASELINE


,Method,Precision@50
0,Week-4 baseline,0.32
1,Logistic Regression,0.70



Difference: 0.38


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Errors and interpretation

On the held-out grouped test set, Logistic Regression achieved a Precision@50 of 0.70 compared with 0.32 for the Week-4 baseline, an observed difference of 0.38 on this split.

This suggests that combining multiple early-window search and engagement signals produced a more useful ranking of later-declining pages than the simple rule-based baseline for this test set.

However, the result is directional rather than causal. The model can still produce false positives and false negatives, and performance may vary across clients or time periods. The grouped split reduces client overlap, but further validation is needed before treating the result as generalisable.

In [9]:
# Predictions at 0.5 threshold for error analysis
test_df["prediction"] = (
    test_df["model_score"] >= 0.5
).astype(int)

# Error types
test_df["error_type"] = np.select(
    [
        (test_df["prediction"] == 1) & (test_df["declining"] == 0),
        (test_df["prediction"] == 0) & (test_df["declining"] == 1)
    ],
    [
        "False Positive",
        "False Negative"
    ],
    default="Correct"
)

print("ERROR COUNTS")
print(test_df["error_type"].value_counts())

# Logistic Regression coefficients
coef = model.named_steps["logreg"].coef_[0]

importance = pd.DataFrame({
    "feature": features,
    "coefficient": coef,
    "absolute_importance": np.abs(coef)
}).sort_values(
    "absolute_importance",
    ascending=False
)

print("\nFEATURE INTERPRETATION")
display(importance)

# Top false positives and false negatives
false_positives = (
    test_df[test_df["error_type"] == "False Positive"]
    .sort_values("model_score", ascending=False)
    .head(10)
)

false_negatives = (
    test_df[test_df["error_type"] == "False Negative"]
    .sort_values("model_score", ascending=True)
    .head(10)
)

safe_cols = [
    "content_hash_id",
    "impressions",
    "clicks",
    "avg_position",
    "pageviews",
    "engaged_sessions",
    "model_score",
    "declining"
]

print("\nTOP FALSE POSITIVES")
display(false_positives[safe_cols])

print("\nTOP FALSE NEGATIVES")
display(false_negatives[safe_cols])

ERROR COUNTS
error_type
Correct           758
False Negative    321
False Positive     88
Name: count, dtype: int64

FEATURE INTERPRETATION


,feature,coefficient,absolute_importance
1,clicks,1.114673,1.114673
2,avg_position,-0.114306,0.114306
4,engaged_sessions,0.074846,0.074846
0,impressions,0.074793,0.074793
3,pageviews,0.023384,0.023384



TOP FALSE POSITIVES


,content_hash_id,impressions,clicks,avg_position,pageviews,engaged_sessions,model_score,declining
23406,content_a47fd3545d1a68c6,2678,45,8.903121,102.0,2.0,0.914311,0
23360,content_30352f0b93c8b98d,1600,34,17.999995,62.0,4.0,0.854179,0
23388,content_7b50820e76b1a006,886,36,29.122832,68.0,1.0,0.831766,0
16414,content_f93cf81d5564765d,701,25,3.242482,26.0,0.0,0.763085,0
23441,content_ed1bd5af6c187f6f,1928,20,19.261200,54.0,5.0,0.745533,0
15616,content_7020c8f8454d0b0a,1411,18,1.946918,17.0,0.0,0.694803,0
15480,content_3a26efe9c2ea74b3,1915,15,3.142378,17.0,0.0,0.660314,0
15569,content_5b0577ed537797f2,2482,13,6.164683,18.0,1.0,0.645451,0
23423,content_c1a10050d6a3c045,677,13,4.907744,24.0,0.0,0.627185,0
23361,content_369b08251e576487,383,12,7.277130,17.0,1.0,0.620083,0



TOP FALSE NEGATIVES


,content_hash_id,impressions,clicks,avg_position,pageviews,engaged_sessions,model_score,declining
23348,content_0c29f92ee0a6987f,6,1,94.333333,2.0,0.0,0.279744,1
15040,content_4c7322376862be65,13,1,66.769231,1.0,0.0,0.332327,1
15259,content_c5c04445dec4a946,49,2,70.193333,2.0,0.0,0.337499,1
15937,content_f5c1fb735fb4a08c,7,1,59.000000,1.0,0.0,0.348015,1
23437,content_e4a6ed9b9592cb9c,6,1,56.000000,1.0,0.0,0.354170,1
23427,content_c8e3854bc9ebd8de,2,1,52.000000,2.0,0.0,0.362569,1
18603,content_48413c0bbfed5580,8,2,55.500000,6.0,0.0,0.368031,1
23411,content_adf670cc06cba5e2,67,1,44.656716,2.0,0.0,0.378335,1
15194,content_a23039505d75a509,5,1,33.000000,1.0,0.0,0.402884,1
15078,content_5b990f73c5250a4d,8,1,32.500000,2.0,0.0,0.404119,1


The error analysis found 88 false positives and 321 false negatives at the 0.5 classification threshold. The model therefore misses more declining pages than it incorrectly flags, although the capstone's main ranking metric is Precision@50 rather than threshold-based classification accuracy.

Among the standardised Logistic Regression coefficients, clicks had the largest absolute coefficient (1.115), while average position, engaged sessions, impressions, and pageviews had substantially smaller coefficients. This indicates that the fitted model leaned most strongly on early-window click activity for this split.

Several of the strongest false negatives had very low early-window impressions and clicks and relatively weak average positions. This suggests that low-activity pages are harder for this model to identify as future decline cases. These observations are directional and require further validation across clients and time periods.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.